In [1]:
# %pip install pyspark

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pyspark in c:\Coding\Lib\site-packages (4.1.1)




[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("CA2") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Started")

Spark Session Started


In [ ]:
bronze_df = spark.read.csv("IMDB Dataset.csv",header=True,inferSchema=True)
print("Bronze Layer")
bronze_df.show(5)

Bronze Layer
+--------------------+--------------------+
|              review|           sentiment|
+--------------------+--------------------+
|One of the other ...|            positive|
|"A wonderful litt...| not only is it w...|
|"I thought this w...| but spirited you...|
|Basically there's...|            negative|
|"Petter Mattei's ...| power and succes...|
+--------------------+--------------------+
only showing top 5 rows


In [14]:
silver_df = bronze_df.dropna()
print("Silver Layer")
silver_df.head()

Silver Layer


Row(review="One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is du

In [13]:
gold_df = silver_df.select("review", "sentiment")
print("Gold Layer")
gold_df.head()

Gold Layer


Row(review="One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is du

In [6]:
from pyspark.ml.feature import Tokenizer
from pyspark.ml.feature import HashingTF
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline


In [ ]:
small_df = gold_df.limit(1000)
tokenizer = Tokenizer(inputCol="review",outputCol="words")

hashingTF = HashingTF(inputCol="words",outputCol="features",numFeatures=1000)

label_indexer = StringIndexer(inputCol="sentiment",outputCol="label")

lr = LogisticRegression(featuresCol="features",labelCol="label")

pipeline = Pipeline(stages=[tokenizer, hashingTF, label_indexer, lr])

model = pipeline.fit(small_df)
result = model.transform(small_df)
print("ML Pipeline Output")

ML Pipeline Output


In [12]:
result.select("review","sentiment","prediction").show(10)

+--------------------+--------------------+----------+
|              review|           sentiment|prediction|
+--------------------+--------------------+----------+
|One of the other ...|            positive|       0.0|
|"A wonderful litt...| not only is it w...|     257.0|
|"I thought this w...| but spirited you...|     158.0|
|Basically there's...|            negative|       1.0|
|"Petter Mattei's ...| power and succes...|     274.0|
|"Probably my all-...| but that only ma...|     160.0|
|I sure would like...|            positive|       0.0|
|This show was an ...|            negative|       1.0|
|Encouraged by the...|            negative|       1.0|
|If you like origi...|            positive|       0.0|
+--------------------+--------------------+----------+
only showing top 10 rows


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Evaluate the model using the accuracy metric
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(result)
print(f"Model Accuracy: {accuracy:.4f}")
